## Class Defining a One-Dimensional Cartesian Grid

In order to solve any sort of problem using the finite volume method, we need to start with a grid on which we discretize our equations. As such, any computational code typically must start by defining some sort of data structure to store the grid quantities, along with code to calculate relevant grid parameters. The problem presently being solved requires just a simple one-dimensional Cartesian grid, as implemented in the code cell below. 

In Python, every class must include a method called ``__init__``, which is invoked any time a new instance of the class is created. This method is commonly called a *constructor*, since it is used to construct a new object.

<div class="alert alert-success">

**Tip:** Review the structure of [Python classes](https://docs.python.org/3.6/tutorial/classes.html) by reading sections 9.1-9.3 in the linked article.

</div>

The constructor for the class ``Grid`` takes four arguments, as listed in the following table.

Variable    | Description
:----------:| :------------------------: 
`lx`        | total length of domain in x-direction [m]
`ly`        | total length of domain in y-direction [m]
`lz`        | total length of domain in z-direction [m]
`ncv`       | number of control volumes in domain

These variables must be provided by the user of the class. The nomenclature implemented in this class (and others to follow) is to denote private member variables with preceding underscores. Although Python does not prevent access to these private member variables like other compiled languages (e.g. C++), it is a good practice to mark them as such, to warn users of the class that they should not be accessing these members (risking unexpected behaviour if they do). Also, this class has several methods marked with the [`@property` decorator](https://docs.python.org/3/library/functions.html#property). This allows users of the class to read the private member variables with controlled access. Here, no methods are provided to set these values, only those to get them. This is because all mesh geometry is calculated internally and the user should never change this, but they do require the ability to access it.

A summary of the variables provided by this class is:

Variable    | Description                                         | Array dimension
:----------:| :--------------------------------------------------:|:---------------:
`ncv`       | number of control volumes in domain                 | N/A
`xf`        | array of x locations of face integration points [m] | `ncv+1`
`xP`        | array of x locations of cell centroids [m]          | `ncv+2`
`dx_WP`     | array of distances from W cell to P cell [m]        | `ncv`
`dx_PE`     | array of distances from P cell to E cell [m]        | `ncv`
`Af`        | array of face areas [m$^2$]                         | `ncv+1`
`Aw`        | array of west face areas [m$^2$]                    | `ncv`
`Ae`        | array of east areas [m$^2$]                         | `ncv`
`Ao`        | array of outer surface areas [m$^2$]                | `ncv`
`vol`       | array of cell volumes [m$^3$]                       | `ncv`

In general arrays relating to face-based quantities have dimension `ncv+1` (since this includes the boundary faces) and cell-based quantities have dimension `ncv`. The exception is the array `xP` which has dimension `ncv+2` to account for 'virual cells' located on the boundaries with zero thickness.  The virtual cells are employed simply to make interpolations at the boundaries more straightforward, as will be seen in the code. A diagram showing the storage scheme for `xP` including the virtual cells on the boundaries is given below. For convenience, functions are also included to return arrays containing the face areas on the east and west sides of each control volume (i.e. `Aw` and `Ae`). These functions are simply taking data out of `af` and are not strictly needed, but make subsequent more readable. Similarly, functions are provided to return the distances $\Delta x_{WP}$ and $\Delta x_{PE}$.

![VirtualControlVolumes](Figures/2-VirtualControlVolumes.png)

Note that this class makes use of the [numpy](http://www.numpy.org) library, as indicated by the first line, ``import numpy as np``.

<div class="alert alert-success">

**Tip:** Review the [NumPy Quickstart Guide](https://docs.scipy.org/doc/numpy/user/quickstart.html) and keep this link handy for future reference.

</div>


In [ ]:
class Grid:
    def __init__(self, lx, ly, lz, ncv):

## Class Defining the Set of Coefficients

In addition to providing a class that defines the grid, it is useful to implement a class that can be used to store the terms that are used to form the linear system that is to be solved. While raw arrays could be used for this purpose, an object-oriented design is preferred because it allows us to provide some extra functionality, such as the function `zero` which zeros all of the arrays. This design also prevents us from having to pass around a lot of arrays to functions, since all of the arrays are grouped together in this class. Read through the implementation below to understand how the coefficients are stored, how the user can access them, and how they can be modified.

In [ ]:
class ScalarCoeffs:

## Classes Defining the Boundary Conditions

In order to solve any partial differential equation, boundary conditions are required. Three different types of boundary conditions will be considered, namely "Dirichlet", "Neumann", and "Robin". Each of these boundary conditions types will be explained, but first, let us discuss how we identify the boundaries of the domain.

### Enumeration Class Defining the Boundary Locations

The next code cell defines a special kind of class, called an "enumeration" (abbreviated `Enum` in Python). Enumerations help make code more readable by avoiding statements like:

```
if loc == 1:
    do something
elif loc == 2:
    do something else
```

In the example above, it is not clear what the integers `1` and `2` actually represent. To make the code more readable, one could certainly add comments, but these may become out of date if the meanings of these integers were changed in one part of the code and the comments were not updated. It is always better to make code readable by itself and use comments as a secondary method of clarifying the meaning. By using the `BoundaryLocation` class that is written below, we can re-write this as:

```
if loc == BoundaryLocation.WEST:
    do something
elif loc == BoundaryLocation.EAST:
    do something else

```

This code is immediately more clear, and probably needs no further comments to clarify. Internally, enumerations are just integers, but wrapped in such a way that they become more useful.

<div class="alert alert-success">

**Tip:** Review the [Enum class](https://docs.python.org/3.6/library/enum.html) by reading sections 8.13.1-8.13.2 in the linked article.

</div>


In [ ]:
from enum import Enum

class BoundaryLocation(Enum):
    """Enumeration class defining boundary condition locations"""
    WEST = 1
    EAST = 2

### Boundary Condition Types

**Dirichlet Boundary Condition**

The Dirichlet boundary condition specifies the value of the dependent variable at the boundary, i.e.:

$$ T = T_{b} $$

**Neumann Boundary Condition**

The Neumann boundary condition specifies the the diffusive flux at the boundary, which is equivalent to specifying the gradient of the dependent variable at the boundary, i.e.:

$$ \frac{\partial T}{\partial x} = \left.\frac{\partial T}{\partial x}\right|_b = g_b$$

where $g_b$ is the specified gradient at the boundary.

For the west boundary we may write:

$$ \frac{T[1] - T[0]}{\Delta x_{WP}[0]} =  g_b $$

The figure below illustrates in more detail the nomenclature for the temperature and displacement arrays for the west boundary.

![WestBoundary](Figures/2-WestBoundary.png)

The boundary value can then be calculated as:

$$ T[0] = T[1] - g_b \cdot \Delta x_{WP}[0]$$

Note that the sign of $g_b$ is defined such that a gradient in the positive x-direction is defined as positive. For a positive gradient, $T[0] < T[1]$.

For the east boundary we may write: 

$$ \frac{T[-1] - T[-2]}{\Delta x_{PE}[-1]} =  g_b $$

Note that the array index `-1` refers to the last element of the array, `-2` refers to the second last element, etc. The nomenclature for the east boundary is further illustrated in the figure below.

![EastBoundary](Figures/2-EastBoundary.png)

For the east boundary, the boundary value is:

$$ T[-1] = T[-2] + g_b \cdot \Delta x_{PE}[-1] $$

**Robin Boundary Condition**

The Robin boundary condition specifies a boundary flux based on Newton's cooling law applied to the boundary face. For the west boundary, this is written as:

$$ k \frac{T[1] - T[0]}{\Delta x_{WP}[0]} = h \cdot (T[0] - T_\infty) $$

which expresses a balance between conduction in the interior of the domain and convection at the surface, where $h$ is the convection coefficient and $T_\infty$ is the ambient temperature. Note that if $T[1] > T[0]$ heat is leaving the domain by diffusion, and is represented by a positive left side of the equation above. The right side of the equation is positive when $T[0] > T_\infty$, which also represents heat leaving the domain. Therefore the equation above is consistent.

Based in the flux balance above, the boundary value is computed as:

$$ T[0]= T[1] - \Delta x_{WP}[0] \cdot \frac{h}{k} \cdot (T[0] - T_\infty) $$

This expression can then be solved for $T[0]$ so that it can be implemented into the code:

$$ T[0]= \frac{T[1] + \Delta x_{WP}[0] \cdot \frac{h}{k} T_\infty}{1 + \Delta x_{WP}[0] \cdot \frac{h}{k}} $$

<div class="alert alert-info">

**Exercise:** Derive the expression for a Robin boundary condition on the east boundary.

</div>

### Implementation of the Boundary Conditions

When we implement the classes that handle imposing boundary conditions, a lot of effort can be saved by ensuring that they all have a common interface. That will enable us to pass any one of the boundary conditions to another part of the code and not have to consider its type. Here we will ensure that each class implements each of the following member functions:

- `value()`: returns the computed boundary value.
- `coeff()`: returns the linearization coefficient for the boundary value, which is equal to the derivative of the boundary value with respect to the adjacent cell value; this is important when computing the linearization coefficients that appear in the linear system.
- `apply()`: applies the boundary value to the field variable array that is held as a reference within the class.

Each class must also define a constructor, but this does not need to have a common interface. This is one of the advantages of non-typed scripting languages like Python, since any type of object can be passed easily. In C++ this would require the use of inheritance and pointers to implement (not to be further discussed here).

Below are the classes `DirichletBc` and `NeumannBc`.

In [ ]:
class DirichletBc:

In [ ]:
class NeumannBc: